In [1]:
# Parte 4B — Validação Cruzada (k-Fold de 2 até 10)

import torch
import torch.nn as nn
import torch.optim as optim
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from sklearn.preprocessing import StandardScaler
from sklearn.model_selection import StratifiedKFold
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score, confusion_matrix

# Carrega e prepara os dados
df = pd.read_csv('../data/breast_cancer.csv')
df = df.drop(columns=['id'], errors='ignore')
X = df.drop(columns='diagnosis')
y = df['diagnosis'].map({'M': 1, 'B': 0})

scaler = StandardScaler()
X_scaled = scaler.fit_transform(X)
X_scaled = np.nan_to_num(X_scaled, nan=0.0, posinf=1.0, neginf=-1.0)
X_tensor = torch.FloatTensor(X_scaled)
y_tensor = torch.FloatTensor(y.values).reshape(-1, 1)

# Modelo
class NeuralNetwork(nn.Module):
    def __init__(self, input_size):
        super().__init__()
        self.model = nn.Sequential(
            nn.Linear(input_size, 32),
            nn.ReLU(),
            nn.Linear(32, 32),
            nn.ReLU(),
            nn.Linear(32, 1),
            nn.Sigmoid()
        )
    def forward(self, x):
        return self.model(x)

# Função de treino para um fold
def train_fold_model(model, train_loader, val_loader, epochs=100):
    optimizer = optim.Adam(model.parameters(), lr=0.001)
    criterion = nn.BCELoss()
    history = {'loss': [], 'val_loss': []}
    for epoch in range(epochs):
        model.train()
        train_loss = 0
        for xb, yb in train_loader:
            optimizer.zero_grad()
            preds = model(xb)
            yb = yb.float().view(-1, 1)
            loss = criterion(preds, yb)
            loss.backward()
            optimizer.step()
            train_loss += loss.item()
        history['loss'].append(train_loss / len(train_loader))

        model.eval()
        val_loss = 0
        with torch.no_grad():
            for xb, yb in val_loader:
                yb = yb.float().view(-1, 1)
                preds = model(xb)
                loss = criterion(preds, yb)
                val_loss += loss.item()
        history['val_loss'].append(val_loss / len(val_loader))
    return model, history

# Loop de k-folds
for k in range(2, 11):
    print(f"\n🔁 Validação Cruzada com k={k}")
    kf = StratifiedKFold(n_splits=k, shuffle=True, random_state=42)

    accs, precs, recs, f1s = [], [], [], []
    fold = 1

    for train_idx, test_idx in kf.split(X_scaled, y):
        X_train, X_val = X_tensor[train_idx], X_tensor[test_idx]
        y_train, y_val = y_tensor[train_idx], y_tensor[test_idx]

        train_loader = torch.utils.data.DataLoader(torch.utils.data.TensorDataset(X_train, y_train), batch_size=32, shuffle=True)
        val_loader = torch.utils.data.DataLoader(torch.utils.data.TensorDataset(X_val, y_val), batch_size=32)

        model = NeuralNetwork(X_tensor.shape[1])
        model, _ = train_fold_model(model, train_loader, val_loader, epochs=50)

        # Avaliação
        model.eval()
        with torch.no_grad():
            y_pred = model(X_val)
            y_pred_bin = (y_pred > 0.5).float()

        accs.append(accuracy_score(y_val.numpy(), y_pred_bin.numpy()))
        precs.append(precision_score(y_val.numpy(), y_pred_bin.numpy(), zero_division=0))
        recs.append(recall_score(y_val.numpy(), y_pred_bin.numpy(), zero_division=0))
        f1s.append(f1_score(y_val.numpy(), y_pred_bin.numpy(), zero_division=0))

        fold += 1

    # Resultados médios
    print(f"✔️ Resultados Médios para k={k}:")
    print(f"Acurácia média: {np.mean(accs):.4f}")
    print(f"Precisão média: {np.mean(precs):.4f}")
    print(f"Recall médio: {np.mean(recs):.4f}")
    print(f"F1-score médio: {np.mean(f1s):.4f}")


c:\Python310\lib\site-packages\sklearn\utils\extmath.py:1101: RuntimeWarning: invalid value encountered in divide
  updated_mean = (last_sum + new_sum) / updated_sample_count
c:\Python310\lib\site-packages\sklearn\utils\extmath.py:1106: RuntimeWarning: invalid value encountered in divide
  T = new_sum / new_sample_count
c:\Python310\lib\site-packages\sklearn\utils\extmath.py:1126: RuntimeWarning: invalid value encountered in divide
  new_unnormalized_variance -= correction**2 / new_sample_count



🔁 Validação Cruzada com k=2
✔️ Resultados Médios para k=2:
Acurácia média: 0.9737
Precisão média: 0.9804
Recall médio: 0.9481
F1-score médio: 0.9640

🔁 Validação Cruzada com k=3
✔️ Resultados Médios para k=3:
Acurácia média: 0.9701
Precisão média: 0.9723
Recall médio: 0.9483
F1-score médio: 0.9591

🔁 Validação Cruzada com k=4
✔️ Resultados Médios para k=4:
Acurácia média: 0.9771
Precisão média: 0.9857
Recall médio: 0.9528
F1-score médio: 0.9687

🔁 Validação Cruzada com k=5
✔️ Resultados Médios para k=5:
Acurácia média: 0.9667
Precisão média: 0.9595
Recall médio: 0.9530
F1-score médio: 0.9550

🔁 Validação Cruzada com k=6
✔️ Resultados Médios para k=6:
Acurácia média: 0.9702
Precisão média: 0.9726
Recall médio: 0.9479
F1-score médio: 0.9591

🔁 Validação Cruzada com k=7
✔️ Resultados Médios para k=7:
Acurácia média: 0.9806
Precisão média: 0.9817
Recall médio: 0.9667
F1-score médio: 0.9732

🔁 Validação Cruzada com k=8
✔️ Resultados Médios para k=8:
Acurácia média: 0.9772
Precisão média: 0